# **VisDrone İnsan Tespiti Projesi**
## Proje özeti
Bu projede Colab Pro ortamında **VisDrone-DET2019** dataseti kullanılarak drone görüntülerinden insan tespiti amaçlanmıştır.  
**Son doğrulama ölçümleri (548 doğrulama görüntüsü):** Precision 0.7642, Recall 0.6207, mAP@50 0.6910 ve mAP@50–95 0.3278. Bu değerler `best.pt` dosyasının ayrı doğrulamasından alınmıştır; test kümesi sonucu değildir.  
## Kullanılan
**Ortam:** Colab Pro  
**Çalışma Zamanı:** Python 3  
**Donanım:** A100 GPU  
**Model:** YOLO11m-P2  
**Eğitim Parametreleri:** imgsz: 832 | Epoch: 150 | Batch: 24  

---
### Başlamadan Önce
1. `Çalışma Zamanı > Çalışma Zamanı Türünü Değiştir > Çalışma zamanı türü > Python 3` seçili olsun
1. `Çalışma Zamanı > Çalışma Zamanı Türünü Değiştir > Donanım hızlandırıcısı > A100 GPU` seçili olsun
---
### Başlarken
1. Hücreleri **Tümünü çalıştır** seçeneği ile ya da **sırayla** çalıştır: 1 → 2 → 3 → 4
2. Google Drive Bağlantısı için izin istediğinde izin vermeyi unutma yoksa oturum sonlandığında veriler kaybolur
---
| Hücre | Görev |
|---|---|
| 1 | GPU Kontrolü & Kurulum |
| 2 | Veri Düzenleme |
| 3 | Eğitim |
| 4 | Doğrulama |
| 5 | Video Ön İşleme |
| 6 | ByteTrack Çıkarım |

---
# **BÖLÜM 1 - GPU Kontrolü & Kurulum**
---

GPU kontrolü şart eğer sistem CPU ile bu eğitimi yapmaya kalkarsa sorun çıkar.

In [ ]:
import torch

# GPU kontrolü
if not torch.cuda.is_available():
    raise EnvironmentError(
        "GPU bulunamadı! Colab çalışma zamanı ayarlarından GPU seçin."
    )
gpu_name  = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")

#####En başından gerekli kütüphaneleri kuruyoruz ki diğer diğer hücrelerde problem yaşamayalım. Oturum sonlandırıldıktan sonra kütüphane bulunamadı hatası alınırsa buradan tekrar kurulum yapabiliriz.

In [ ]:
# Gerekli kütüphanelerin kurulumu
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics', 'lapx', 'opencv-python', 'matplotlib'],
    check=True
)
print("Kurulum tamamlandı")


#####Finalde elde edeceğimiz çıktıları oturum sonlandığında kaybetmemek için Drive bağlantısı yapıyoruz. Bu adımda bağlantı için izin veriyoruz.

In [ ]:
# Drive bağlantısı
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("Drive başarıyla bağlandı")

#####Dataseti colab'in hafızasına indiriyoruz çünkü Drive'a göre daha hızlı ramde de gereksiz yer kaplamasını istemiyoruz.

In [ ]:
import os

# Dataseti tutacağımız klasörü oluşturuyoruz
os.makedirs('/content/datasets/VisDrone', exist_ok=True)

!wget -q -O /content/datasets/VisDrone/train.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
!unzip -q -o /content/datasets/VisDrone/train.zip -d /content/datasets/VisDrone/
print("Train verisi indiriliyor")

!wget -q -O /content/datasets/VisDrone/val.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
!unzip -q -o /content/datasets/VisDrone/val.zip -d /content/datasets/VisDrone/
print("Validate verisi indiriliyor")

# Yer kaplamaması için zip dosyalarını temizliyoruz
!rm /content/datasets/VisDrone/*.zip

print("Veri seti Colab'e indirildi")

---
# **BÖLÜM 2 - Veri Düzenleme**
---

#####Orjinal Datasette 10 tane sınıf varken bize insan ve yaya sınıfı gerekiyor.Bu bölümde diğer sınıfları atıp insan ve yayayı birleştirerek tek bir 'insan' sınıfı oluşturuyoruz. Ayrıca kalitesiz ve yanıltıcı etiketleri(skoru 0 olanları) temizleyerek veriyi YOLO formatına uygun hale getiriyoruz.

In [ ]:
import os
from PIL import Image

def visdrone_to_yolo_human_only(base_dir):
    images_dir = os.path.join(base_dir, 'images')
    annotations_dir = os.path.join(base_dir, 'annotations')
    labels_dir = os.path.join(base_dir, 'labels')

    if not os.path.exists(annotations_dir): return
    os.makedirs(labels_dir, exist_ok=True)
    print(f"İşleniyor: {base_dir}")

    file_count = 0
    for img_name in os.listdir(images_dir):
        if not img_name.endswith('.jpg'): continue

        ann_path = os.path.join(annotations_dir, img_name.replace('.jpg', '.txt'))
        label_path = os.path.join(labels_dir, img_name.replace('.jpg', '.txt'))

        if not os.path.exists(ann_path): continue

        with Image.open(os.path.join(images_dir, img_name)) as img:
            img_width, img_height = img.size

        new_lines = []
        with open(ann_path, 'r', encoding='utf-8') as f:
            for line in f.readlines():
                parts = line.strip().split(',')
                if len(parts) < 8: continue

                # Yoksayılacak bölgeleri atla(score=0 olanları)
                score = int(parts[4])
                if score == 0: continue

                # 1=Yaya, 2=İnsanlar olarak sınıflandırıyoruz
                category = int(parts[5])
                if category in [1, 2]:
                    x_center = (float(parts[0]) + float(parts[2]) / 2.0) / img_width
                    y_center = (float(parts[1]) + float(parts[3]) / 2.0) / img_height
                    w = float(parts[2]) / img_width
                    h = float(parts[3]) / img_height
                    # Sınıfı 0 yapıp kaydediyoruz
                    new_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

        if len(new_lines) > 0:
            with open(label_path, 'w', encoding='utf-8') as f:
                f.writelines(new_lines)
            file_count += 1

    print(f"{file_count} görsel başarıyla filtrelendi ve YOLO formatına çevrildi.")

# Fonksiyonu çalıştır
visdrone_to_yolo_human_only('/content/datasets/VisDrone/VisDrone2019-DET-train')
visdrone_to_yolo_human_only('/content/datasets/VisDrone/VisDrone2019-DET-val')

#####Modelin eğitim verilerinin nerede olacağını ve hangi sınıfı arayacağını bilmesi için rehberlik edecek YAML dosyası oluşturuyoruz. Bu dosyanın kalıcı olması için de drive'a kaydediyoruz.

In [ ]:
import os

# YAML dosyasını oluşturuyoruz
yaml_content = """
path: /content/datasets/VisDrone
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
nc: 1
names:
  0: Human
"""

# YAML dosyasını drive'a kaydediyoruz
hedef_klasor = '/content/drive/MyDrive/Tez_YOLO11_Projesi'
os.makedirs(hedef_klasor, exist_ok=True)

yaml_yolu = os.path.join(hedef_klasor, 'custom_visdrone.yaml')
with open(yaml_yolu, 'w') as f:
    f.write(yaml_content.strip())
print("custom_visdrone.yaml dosyası hazır")

#####YOLO11m modeli standart olarak P3, P4 ve P5 katmanlarını kullanıyor. Bu katmanlar sırasıyla 8x8, 16x16 ve 32x32 piksellik alanları tek bir hücre gibi analiz ediyor. Ancak bizim VisDrone veri setindeki insanlar drone kamerasından bakınca çok küçük kaldığı için, standart P3-P5 mimarisi yetersiz kalıyor. Bu yüzden, çok daha küçük alanlara (4x4) odaklanıp detayları yakalayabilen P2 katmanını da mimariye dahil ettik.

In [ ]:
yaml_content = """# Ultralytics YOLO, AGPL-3.0 license
# YOLO11 object detection model with P2-P5 outputs.

nc: 1
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]          # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]          # 16 (P3/8)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]]          # 19 (P2/4)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]          # 22 (P3/8)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]          # 25 (P4/16)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]          # 28 (P5/32)

  - [[19, 22, 25, 28], 1, Detect, [nc]]
"""

with open('yolo11m-p2.yaml', 'w') as f:
    f.write(yaml_content)

print("yolo11m-p2.yaml oluşturuldu")



---
# **BÖLÜM 3 - Eğitim**
---

#####Burada modeli eğiticez. Dronelar için kullanılabilecek en iyi modeli YOLO11m olarak seçiyoruz. Standart kullanılan katmanların yetersiz olduğunu düşünerek P2 katmanını ekliyoruz ve gerekli ince ayarları yapıyoruz.

In [ ]:
BATCH      = 24 # GPU tek seferde 24 görüntü birden gösterecek
WORKERS    = 12 # arka planda çalışacak olan işlemci sayısı
CACHE      = 'ram' # veri setini rame yazıcaz ki daha hızlı işlem yapalım
YAML_PATH  = '/content/drive/MyDrive/Tez_YOLO11_Projesi/custom_visdrone.yaml' #modele rehberlik edecek YAML dosyası
OUTPUT_DIR = '/content/drive/MyDrive/Tez_YOLO11_Projesi' #eğitim bittiğinde çıktının kaydedileceği yol

import os

from ultralytics import YOLO
# P2 katmanını burada kullanıcaz
model = YOLO('yolo11m-p2.yaml').load('yolo11m.pt')
print("Model yüklendi")
model.info()

print(f"\n Eğitim başlıyor batch={BATCH} | imgsz=832 | cache='{CACHE}'")
print("─" * 60)


# eğitim için gerekli parametreler

results = model.train(
    data     = YAML_PATH,   # YAML dosyası
    epochs   = 150,         # eğitim 150 turluk olacak
    imgsz    = 832,         # görüntü 832x832 olarak işlenecek
    batch    = BATCH,       # A100 için 24 olarak belirledik(A100 kullanamıza rağmen eğitim sırasında 3 kere GPU rami konusunda problem yaşadık)
    device   = 0,           # eğitimde GPU(A100) kullanıcaz
    workers  = WORKERS,     # işlemci sayısı
    project  = OUTPUT_DIR,  # çıktı yolu
    name     = 'YOLO11m_P2_832_Final', # tüm sonuçların kaydedileceği klasörün adı
    exist_ok = True, # mevcutta aynı klasör varsa üstüne yazıcaz

    cache    = CACHE,

    # görüntülere uygulayacağımız ayarlar
    hsv_h        = 0.015,   # renk tonunu çok az değiştir(farklı lens/kamera efekti)
    hsv_s        = 0.7,     # renk doygunluğunu değiştir
    hsv_v        = 0.4,     # parlaklığı değiştir
    degrees      = 0.0,     # rotasyonu 0 yapıyoruz dik açıdan çekiyoruz
    translate    = 0.1,     # droneun salınım hareketini taklit et(görüntüyü çok az kaydır)
    scale        = 0.2,     # küçük nesne piksellerini koru
    fliplr       = 0.5,     # görüntüyü yatay çevirme
    flipud       = 0.0,     # görüntüyü dikey çevirme kapalı drone görüntüsünde gereksiz
    mosaic       = 1.0,     # her epochda 4 farklı drone görüntüsünü alıp kesip biçip tek bir ekranda birbirine yapıştırıyoruz
    copy_paste   = 0.2,     # insanları kopyala görüntünün başka kısımlarına yapıştır
    mixup        = 0.05,    # iki resmi üst üste bindir komplike sahneler yarat
    erasing      = 0.0,     # küçük nesneleri silme kapalı
    close_mosaic = 10,      # son 10 tur mosaic efektini kapat

    # optimizasyon ayarları
    optimizer     = 'AdamW',  # ezberi önleyen algoritma tercihi
    lr0           = 0.003,    # en yüksek hedefi amaçlayan başlangıç öğrenme oranı(learning rate)
    lrf           = 0.01,     # eğitim sonunda lr0 ile çarpılacak katsayı(nihai öğrenme oranı)
    warmup_epochs = 3,        # ilk 3 tur ısınma turu olsun
    weight_decay  = 0.0005,   # aşırı öğrenmeyi engelleyen ceza katsayısı

    # kayıp oranları
    box = 7.5,  # yeşil kutunun hedefe tam oturması için ceza oranı
    cls = 0.5,  # sınıflandırma detayı(sadece insan aradığımız için düşük)
    dfl = 1.5,  # küçük nesnelerin kenarlarını daha iyi yakalayabilmek için kenar oranı

    # FP16 mimarisi
    amp = True, # modelin doğruluğunu bozmadan bellek kullanımı yarıya indirip hızı 2 katına çıkaran FP16 mimarisi kullanıcaz

    # güvenlik ayarları
    patience    = 40, # eğitim 40 tur boyunca aynı sonuçları verirse eğitimi durdur
    save_period = 10, # 10 epoch'da bir sonuçları kaydet
    plots       = True, # başarı grafiklerini otomatik çiz
    verbose     = True, # eğitim sürerken arka planda dönen işlemleri ve ilerlemeyi konsola yaz
)

print("\n" + "═" * 60)
print("EĞİTİM TAMAMLANDI")
best = os.path.join(OUTPUT_DIR, 'YOLO11m_P2_832_Final', 'weights', 'best.pt')
if os.path.exists(best):
    print(f"best.pt → {best} ({os.path.getsize(best)/1e6:.1f} MB)")
print("═" * 60)

---
# **BÖLÜM 4 - Doğrulama**
---

#####Bu kısımda eğittiğimiz modelin doğrulamasını yapıyoruz

In [ ]:
!pip install ultralytics -q

In [ ]:
BATCH      = 24
OUTPUT_DIR = '/content/drive/MyDrive/Tez_YOLO11_Projesi'
YAML_PATH  = '/content/drive/MyDrive/Tez_YOLO11_Projesi/custom_visdrone.yaml'

import os,json
from ultralytics import YOLO

# eğitim sonucu oluşan en iyi ağırlık dosyasını çağırıyoruz
BEST_PT = os.path.join(OUTPUT_DIR, 'YOLO11m_P2_832_Final', 'weights', 'best.pt')

if not os.path.exists(BEST_PT):
    print(f"best.pt bulunamadı: {BEST_PT}. Önce BÖLÜM 3'ü (eğitimi) tamamlayın.")
else:
    model = YOLO(BEST_PT)
    metrics = model.val(
        data    = YAML_PATH,
        imgsz   = 832,
        batch   = BATCH,
        device  = 0,
        split   = 'val',  # modeli daha önce görmediği görüntülerle sınava sokuyoruz
        plots   = True,
        verbose = True,
    )
    print("\n" + "═" * 50)
    print("SONUÇLAR")
    print("═" * 50)
    print(f"  mAP@0.50      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.50:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision     : {metrics.box.mp:.4f}")
    print(f"  Recall        : {metrics.box.mr:.4f}")

    results_dict = {
        "mAP50":      round(metrics.box.map50, 4),
        "mAP50-95":   round(metrics.box.map, 4),
        "Precision":  round(metrics.box.mp, 4),
        "Recall":     round(metrics.box.mr, 4),
    }

    save_path = os.path.join(OUTPUT_DIR, 'val_results.json')
    with open(save_path, 'w') as f:
        json.dump(results_dict, f, indent=2)

---
# **BÖLÜM 5 — Video Ön İşleme**
---

####**BU BÖLÜM ZORUNLU DEĞİLDİR AMA YAPILMASI ÖNERİLİR**
#####Test edeceğimiz videoları ön işleyerek opencv'nin istediği formata çeviriyoruz. Bazı videoların formatları opencv'ye problem çıkardığı için bu bölümde video formatını uygun hale çeviriyoruz.

In [ ]:
# test videosunu sol menüdeki dosyalar kısmına "test_video.mp4" adıyla yüklüyoruz
import os, subprocess

VIDEO_INPUT = '/content/test_video.mp4'    # test videosunun yolu
VIDEO_FIXED = '/content/test_video_fixed.mp4' # formatı düzenlenmiş videonun yolu

if not os.path.exists(VIDEO_INPUT):
    print(f"Video bulunamadı: {VIDEO_INPUT}")
else:
    print(f"Kaynak: {VIDEO_INPUT}")

    result = subprocess.run([
        'ffmpeg', '-y', '-i', VIDEO_INPUT,
        '-c:v', 'libx264', '-crf', '23', '-preset', 'fast', # görüntü kalitesini bozmadan formatı h264e çevir
        '-c:a', 'aac', '-b:a', '128k',  # ses formatını standart hale getiriyoruz(aac)
        '-movflags', '+faststart',  # video indexini en başa al (model ilk kareden işlemeye başlayabilsin diye)
        VIDEO_FIXED
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print(f"Dönüşüm tamamlandı: {VIDEO_FIXED}")
    else:
        print(f"FFmpeg hatası:\n{result.stderr[-2000:]}")

---
# **BÖLÜM 6 — ByteTrack ile Çıkarım**
---

#####Takip algoritması için ayarları standart kullanmak yerine kendimiz oluşturuyoruz

In [ ]:
import yaml

custom_tracker = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.4,   # ilk kitlenme katsayısı
    "track_low_thresh": 0.1,    # %10un altına düşene kadar idyi silme
    "new_track_thresh": 0.5,    # yeni id atarken modelin min %50 emin olması lazım
    "track_buffer": 90,         # görüntüde kaybolan insanları 90 kare(30fps için 3 saniye) hafızada tut
    "match_thresh": 0.8         # önceki ve sonraki karedeki kutular arasındaki maksimum fark sınırı
}

with open('ozel_bytetrack.yaml', 'w') as f:
    yaml.dump(custom_tracker, f)

print("ozel_bytetrack.yaml oluşturuldu")

#####Modelimizin insanları takip edip ID atayacağı kısım burası. Bazı ayarlar gerçek hayatta kullanıma uygun değil o yüzden notlar kısmında nelerin değişebileceği belirtilmiştir.

In [ ]:
# Bu prototip mevcut ortamda simüle edebilebilmek için yaratılmıştır
# Daha iyi performans için kodlar C++ diline taşınmalıdır
# Gerçek hayatta gömülü sistem entegrasyonu için model TensorRT (INT8) formatına derlenmelidir(biz FP16 kullanıyoruz)

import cv2, os, random
from ultralytics import YOLO
from datetime import datetime

# dosya yolları
MODEL_PATH = os.path.join(OUTPUT_DIR, 'YOLO11m_P2_832_Final', 'weights', 'best.pt')
VIDEO_IN   = '/content/test_video_fixed.mp4'
OUTPUT_VID = os.path.join(OUTPUT_DIR, 'Video_Sonuclari', 'Tez_ByteTrack_Sonuc.mp4')
os.makedirs(os.path.dirname(OUTPUT_VID), exist_ok=True)

for label, path in [('Model', MODEL_PATH), ('Video', VIDEO_IN)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f" {label} bulunamadı: {path}")

# video ve model hazırlığı
model = YOLO(MODEL_PATH)
cap   = cv2.VideoCapture(VIDEO_IN)
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS   = int(cap.get(cv2.CAP_PROP_FPS)) or 30
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {W}×{H} @ {FPS} FPS — {TOTAL} kare")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VID, fourcc, FPS, (W, H))
if not out.isOpened():
    raise RuntimeError("VideoWriter açılamadı")

random.seed(42)
PALETTE = {i: (random.randint(50,255), random.randint(50,255), random.randint(50,255))
           for i in range(1000)}

print("\nByteTrack başlıyor")
print("─" * 60)

t0      = datetime.now()
n_frame = 0
id_set  = set()

# kare izleme takip döngüsü
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # modelin hedefi tespit edip id atandığı kısım
    res = model.track(
        frame,
        conf    = 0.20, # ağaç arkasın kalan veya benzeri durumda olan insanları kaybetmemesi için conf değerini düşük tuttuk
        iou     = 0.30, # aynı insanı iki kere id atamaması için iç içe geçmiş iki kutu %30dan fazla örtüşüyorsa düşük puanlı olanı sil
        tracker = 'ozel_bytetrack.yaml', # özel ayarlarımızı kullan
        persist = True, # idleri hafızada tut her karede yeniden id üretme
        verbose = False,
        imgsz   = 832,
        device  = 0, # GPU kullan
        augment=True, # görüntüyü 4 farklı açıyla oluşturup hedefi arar gerçek hayatta hız düşüşü yaşamamak için bunun kapatılması gerekir
    )

    if res[0].boxes.id is not None:
        boxes  = res[0].boxes.xyxy.cpu().numpy()
        ids    = res[0].boxes.id.int().cpu().tolist()
        scores = res[0].boxes.conf.cpu().tolist()
        for box, tid, sc in zip(boxes, ids, scores):
            x1, y1, x2, y2 = map(int, box)
            cx, cy = (x1+x2)//2, (y1+y2)//2
            color  = PALETTE[tid % 1000]
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 1)
            cv2.circle(frame, (cx,cy), 3, color, -1)  # hedef merkez noktası
            cv2.putText(frame, f"ID:{tid} {sc:.2f}", (x1, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1, cv2.LINE_AA)
            id_set.add(tid)

    out.write(frame)
    n_frame += 1
    if n_frame % 100 == 0:
        elapsed = max((datetime.now()-t0).seconds, 1)
        fps_avg = n_frame / elapsed
        remain  = (TOTAL - n_frame) / fps_avg
        print(f"  Kare {n_frame}/{TOTAL} | {fps_avg:.1f} FPS | ~{remain:.0f}s kaldı")

cap.release()
out.release()

sure = (datetime.now()-t0).total_seconds()
print("\n" + "═"*60)
print("ÇIKARIM TAMAMLANDI")
print(f"  İşlenen kare  : {n_frame}")
print(f"  Benzersiz ID  : {len(id_set)}")
print(f"  Toplam süre   : {sure:.1f} sn  |  Ort. {n_frame/sure:.1f} FPS")
print(f"  Çıktı         : {OUTPUT_VID}")
print("═"*60)

## Atıf

VisDrone-DET2019: Zhu, Pengfei, et al. “Detection and Tracking Meet Drones Challenge.” *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 44(11), 7380–7399 (2022). [DOI: 10.1109/TPAMI.2021.3119563](https://doi.org/10.1109/TPAMI.2021.3119563).

Atıf, veri kümesini kullanma veya yeniden dağıtma izni yerine geçmez.